In [19]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [20]:
# Helper functions
from anthropic.types import Message, MessageParam, ToolParam

def add_user_message(messages: list[MessageParam], message: Message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(user_message)


def add_assistant_message(messages: list[MessageParam], message: Message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message
    }
    messages.append(assistant_message)


def chat(
        messages: list[MessageParam],
        system: str | None = None,
        temperature = 1.0,
        stop_sequences: list[str] | None = [],
        tools: list[ToolParam] | None=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message: Message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )

In [21]:
# Tools and Schemas

from anthropic.types import ToolParam
from datetime import datetime, timedelta


def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Return the current local date and time formatted as a string using a Python strftime-compatible format pattern. Use this when the user asks for the current date, current time, or both in a specific format.",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "Optional Python strftime format string used to format the current date and time, for example \"%Y-%m-%d %H:%M:%S\" or \"%d/%m/%Y\".",
                "default": "%Y-%m-%d %H:%M:%S",
                "minLength": 1
            }
        },
        "additionalProperties": False
    },
    "strict": True
})


def add_duration_to_datetime(
    datetime_str,
    duration=0,
    unit="days",
    input_format="%Y-%m-%d"
):
    if not datetime_str:
        raise ValueError("datetime_str cannot be empty")
    if not input_format:
        raise ValueError("input_format cannot be empty")
    if not unit:
        raise ValueError("unit cannot be empty")

    dt = datetime.strptime(datetime_str, input_format)

    unit = unit.lower()
    if unit == "days":
        delta = timedelta(days=duration)
    elif unit == "hours":
        delta = timedelta(hours=duration)
    elif unit == "minutes":
        delta = timedelta(minutes=duration)
    elif unit == "seconds":
        delta = timedelta(seconds=duration)
    elif unit == "weeks":
        delta = timedelta(weeks=duration)
    else:
        raise ValueError("unit must be one of: days, hours, minutes, seconds, weeks")

    new_dt = dt + delta
    return new_dt.strftime("%A, %B %d, %Y %I:%M:%S %p")

add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Parse an input datetime string using a Python strptime-compatible format, add a duration in the specified unit, and return the resulting datetime formatted as '%A, %B %d, %Y %I:%M:%S %p'. Use this when the user wants to shift a date or datetime by a number of days, hours, minutes, seconds, or weeks.",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input date or datetime string to parse, for example '2026-04-23' or '2026-04-23 14:30:00' depending on input_format.",
                "minLength": 1
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add. May be positive, zero, or negative.",
                "default": 0
            },
            "unit": {
                "type": "string",
                "description": "The unit for the duration to add.",
                "enum": ["days", "hours", "minutes", "seconds", "weeks"],
                "default": "days"
            },
            "input_format": {
                "type": "string",
                "description": "Optional Python strptime format string used to parse datetime_str, for example '%Y-%m-%d' or '%Y-%m-%d %H:%M:%S'.",
                "default": "%Y-%m-%d",
                "minLength": 1
            }
        },
        "required": ["datetime_str"],
        "additionalProperties": False
    },
    "strict": True
}


def set_reminder(content, timestamp):
    if not content:
        raise ValueError("content cannot be empty")
    if not timestamp:
        raise ValueError("timestamp cannot be empty")

    return {
        "status": "success",
        "message": f"Reminder set for {timestamp}: {content}",
        "content": content,
        "timestamp": timestamp
    }

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Set a reminder with some content and a timestamp. Use this when the user wants to schedule a reminder for a specific date and time.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The reminder text or message to remember.",
                "minLength": 1
            },
            "timestamp": {
                "type": "string",
                "description": "The date and time for the reminder, preferably in ISO 8601 format such as '2026-04-23T18:30:00'.",
                "minLength": 1
            }
        },
        "required": ["content", "timestamp"],
        "additionalProperties": False
    },
    "strict": True
}

In [22]:
from typing import Any
from anthropic.types import ToolUseBlock, ToolResultBlockParam
import json

def run_tool(tool_name: str, tool_input: dict[str, Any]):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)
    raise ValueError(f"Could not invoke given tool_name: {tool_name}")


def run_tools(message: Message) -> list[ToolResultBlockParam]:
    tool_requests: list[ToolUseBlock] = [
        block for block in message.content if block.type == "tool_use"
    ]
    tool_result_blocks: list[ToolResultBlockParam] = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }
        
        tool_result_blocks.append(tool_result_block)
    
    return tool_result_blocks

def run_conversation(messages: list[MessageParam]):
    while True:
        response = chat(messages, tools=[
            get_current_datetime_schema,
            add_duration_to_datetime_schema,
            set_reminder_schema,
        ])

        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)
    
    return messages

In [23]:
messages: list[MessageParam] = []

add_user_message(
    messages,
    "Set a reminder for my doctor's appointment. It's 177 days after Jan. 1st, 2027"
)

run_conversation(messages)

I'll calculate the date that is 177 days after January 1st, 2027, and then set a reminder for you.
Now I'll set a reminder for your doctor's appointment:
Perfect! I've set a reminder for your doctor's appointment on **Sunday, June 27, 2027** at 12:00 AM. This is 177 days after January 1st, 2027.


[{'role': 'user',
  'content': "Set a reminder for my doctor's appointment. It's 177 days after Jan. 1st, 2027"},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll calculate the date that is 177 days after January 1st, 2027, and then set a reminder for you.", type='text'),
   ToolUseBlock(id='toolu_01JCVQhfXuv6FHszjVS3C4bS', caller=DirectCaller(type='direct'), input={'datetime_str': '2027-01-01', 'duration': 177, 'unit': 'days'}, name='add_duration_to_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01JCVQhfXuv6FHszjVS3C4bS',
    'content': '"Sunday, June 27, 2027 12:00:00 AM"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="Now I'll set a reminder for your doctor's appointment:", type='text'),
   ToolUseBlock(id='toolu_01LYGAoJ6yAASj6Yd5VfdpS3', caller=DirectCaller(type='direct'), input={'content': "Doctor's appointment", 'timestamp': '2027-06-27T12:0